# 05 — Feature Selection
### Student Performance Analysis System (SPAS)

This notebook determines which features enter the ML models using three independent methods:
1. Correlation with target
2. Random Forest feature importance
3. Mutual Information

**Input:**  `data/features/dataset_v1_features.csv`  
**Output:** `models/feature_list.pkl` + `data/features/dataset_v1_selected.csv`

## 1. Imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split

os.makedirs('../models', exist_ok=True)
os.makedirs('../eda',    exist_ok=True)

RANDOM_SEED = 42
print("Imports done.")

Imports done.


## 2. Load Dataset & Define Feature Pool

In [5]:
df = pd.read_csv('../data/features/student_performance_dataset_feature_engineering.csv')
print(f"Shape : {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape : (5000, 17)
Columns: ['student_id', 'attendance_percentage', 'quiz_score_avg', 'assignment_score_avg', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'participation_score', 'subject_difficulty_score', 'semester_number', 'final_score', 'pass_fail', 'attendance_tier', 'gpa_tier', 'ca_avg', 'rule_risk_score', 'midterm_relative']


In [6]:
# ── Columns that must NEVER enter the model ───────────────────────────────────
EXCLUDE_COLS = ['student_id', 'final_score', 'pass_fail']

# ── Candidate features — everything else ─────────────────────────────────────
CANDIDATE_FEATURES = [
    'attendance_percentage',
    'quiz_score_avg',
    'assignment_score_avg',
    'midterm_score',
    'historical_gpa',
    'study_hours_per_week',
    'participation_score',
    'subject_difficulty_score',
    'semester_number',
    'attendance_tier',
    'gpa_tier',
    'ca_avg',
    'midterm_relative',
    'rule_risk_score',
]

# ── Targets ───────────────────────────────────────────────────────────────────
y_reg = df['final_score']          # regression target
y_clf = df['pass_fail']            # classification target
X     = df[CANDIDATE_FEATURES]

print(f"\nCandidate features ({len(CANDIDATE_FEATURES)}):")
for f in CANDIDATE_FEATURES:
    print(f"  - {f}")

print(f"\nExcluded columns : {EXCLUDE_COLS}")
print(f"Regression target: final_score")
print(f"Classification target: pass_fail")


Candidate features (14):
  - attendance_percentage
  - quiz_score_avg
  - assignment_score_avg
  - midterm_score
  - historical_gpa
  - study_hours_per_week
  - participation_score
  - subject_difficulty_score
  - semester_number
  - attendance_tier
  - gpa_tier
  - ca_avg
  - midterm_relative
  - rule_risk_score

Excluded columns : ['student_id', 'final_score', 'pass_fail']
Regression target: final_score
Classification target: pass_fail


## 3. Train / Test Split
One split used by all three methods for consistency.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=RANDOM_SEED
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (4000, 14)
X_test  : (1000, 14)
y_train : (4000,)
y_test  : (1000,)


## 4. Method 1 — Correlation with Target

In [8]:
# Absolute Pearson correlation of each feature against final_score
corr_scores = X.corrwith(y_reg).abs().sort_values(ascending=False)

print("── Method 1: Absolute Correlation with final_score ──")
for feat, val in corr_scores.items():
    flag = '✅' if val >= 0.10 else '❌ BELOW THRESHOLD'
    print(f"  {feat:<30}: r = {val:.4f}  {flag}")

# Rank (1 = best)
corr_ranks = corr_scores.rank(ascending=False).astype(int)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v >= 0.10 else 'lightcoral' for v in corr_scores.values]
ax.barh(corr_scores.index[::-1], corr_scores.values[::-1], color=colors[::-1])
ax.axvline(0.10, color='red', linestyle='--', linewidth=1.5, label='Threshold (0.10)')
ax.set_xlabel('Absolute Correlation with final_score')
ax.set_title('Method 1 — Feature Correlation Ranking', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../eda/feature_correlation_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved → eda/feature_correlation_ranking.png")

── Method 1: Absolute Correlation with final_score ──
  ca_avg                        : r = 0.8667  ✅
  rule_risk_score               : r = 0.8565  ✅
  midterm_score                 : r = 0.8187  ✅
  assignment_score_avg          : r = 0.8177  ✅
  quiz_score_avg                : r = 0.8163  ✅
  midterm_relative              : r = 0.7533  ✅
  historical_gpa                : r = 0.6718  ✅
  gpa_tier                      : r = 0.6055  ✅
  study_hours_per_week          : r = 0.5581  ✅
  attendance_percentage         : r = 0.3175  ✅
  attendance_tier               : r = 0.3013  ✅
  subject_difficulty_score      : r = 0.1414  ✅
  participation_score           : r = 0.0909  ❌ BELOW THRESHOLD
  semester_number               : r = 0.0048  ❌ BELOW THRESHOLD

Plot saved → eda/feature_correlation_ranking.png


## 5. Method 2 — Random Forest Feature Importance

In [9]:
print("Training Random Forest (this takes ~10 seconds)...")

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_importance = pd.Series(
    rf.feature_importances_,
    index=CANDIDATE_FEATURES
).sort_values(ascending=False)

print("\n── Method 2: Random Forest Feature Importance ──")
for feat, val in rf_importance.items():
    print(f"  {feat:<30}: importance = {val:.4f}")

# Rank
rf_ranks = rf_importance.rank(ascending=False).astype(int)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rf_importance.index[::-1], rf_importance.values[::-1], color='seagreen')
ax.set_xlabel('Feature Importance Score')
ax.set_title('Method 2 — Random Forest Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('../eda/random_forest_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved → eda/random_forest_importance.png")

Training Random Forest (this takes ~10 seconds)...

── Method 2: Random Forest Feature Importance ──
  ca_avg                        : importance = 0.6462
  rule_risk_score               : importance = 0.1809
  study_hours_per_week          : importance = 0.0310
  midterm_score                 : importance = 0.0287
  subject_difficulty_score      : importance = 0.0286
  participation_score           : importance = 0.0199
  midterm_relative              : importance = 0.0139
  assignment_score_avg          : importance = 0.0134
  quiz_score_avg                : importance = 0.0106
  historical_gpa                : importance = 0.0099
  attendance_percentage         : importance = 0.0096
  semester_number               : importance = 0.0059
  gpa_tier                      : importance = 0.0008
  attendance_tier               : importance = 0.0006

Plot saved → eda/random_forest_importance.png


## 6. Method 3 — Mutual Information

In [10]:
print("Computing Mutual Information scores...")

mi_values = mutual_info_regression(
    X_train, y_train,
    random_state=RANDOM_SEED
)

mi_scores = pd.Series(
    mi_values,
    index=CANDIDATE_FEATURES
).sort_values(ascending=False)

print("\n── Method 3: Mutual Information Scores ──")
for feat, val in mi_scores.items():
    print(f"  {feat:<30}: MI = {val:.4f}")

# Rank
mi_ranks = mi_scores.rank(ascending=False).astype(int)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mi_scores.index[::-1], mi_scores.values[::-1], color='darkorange')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Method 3 — Mutual Information Scores', fontweight='bold')
plt.tight_layout()
plt.savefig('../eda/mutual_information_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved → eda/mutual_information_scores.png")

Computing Mutual Information scores...

── Method 3: Mutual Information Scores ──
  ca_avg                        : MI = 0.7103
  rule_risk_score               : MI = 0.6722
  assignment_score_avg          : MI = 0.5726
  quiz_score_avg                : MI = 0.5637
  midterm_score                 : MI = 0.5351
  midterm_relative              : MI = 0.4301
  historical_gpa                : MI = 0.2995
  study_hours_per_week          : MI = 0.2353
  gpa_tier                      : MI = 0.2227
  attendance_percentage         : MI = 0.0450
  attendance_tier               : MI = 0.0394
  subject_difficulty_score      : MI = 0.0224
  participation_score           : MI = 0.0030
  semester_number               : MI = 0.0000

Plot saved → eda/mutual_information_scores.png


## 7. Consensus Ranking Table

In [11]:
consensus = pd.DataFrame({
    'corr_score'      : corr_scores,
    'corr_rank'       : corr_ranks,
    'rf_importance'   : rf_importance,
    'rf_rank'         : rf_ranks,
    'mi_score'        : mi_scores,
    'mi_rank'         : mi_ranks,
})

consensus['avg_rank'] = (
    consensus['corr_rank'] +
    consensus['rf_rank']   +
    consensus['mi_rank']
) / 3

consensus = consensus.sort_values('avg_rank')

print("── Consensus Feature Ranking Table ──")
print(f"{'Feature':<30} {'CorrRank':>9} {'RF Rank':>8} {'MI Rank':>8} {'AvgRank':>9}")
print("-" * 68)
for feat, row in consensus.iterrows():
    print(f"{feat:<30} {int(row['corr_rank']):>9} "
          f"{int(row['rf_rank']):>8} "
          f"{int(row['mi_rank']):>8} "
          f"{row['avg_rank']:>9.2f}")

── Consensus Feature Ranking Table ──
Feature                         CorrRank  RF Rank  MI Rank   AvgRank
--------------------------------------------------------------------
ca_avg                                 1        1        1      1.00
rule_risk_score                        2        2        2      2.00
midterm_score                          3        4        5      4.00
assignment_score_avg                   4        8        3      5.00
quiz_score_avg                         5        9        4      6.00
midterm_relative                       6        7        6      6.33
study_hours_per_week                   9        3        8      6.67
historical_gpa                         7       10        7      8.00
subject_difficulty_score              12        5       12      9.67
gpa_tier                               8       13        9     10.00
attendance_percentage                 10       11       10     10.33
participation_score                   13        6       13     10

## 8. Select Final Features

Drop features that meet ANY of these criteria:
- Absolute correlation with `final_score` < 0.10
- Consistently in the bottom 3 across all three methods
- Redundant with a stronger already-selected feature

In [12]:
print("── Elimination Analysis ──\n")

# Rule 1: correlation below threshold
low_corr = corr_scores[corr_scores < 0.10].index.tolist()
print(f"Rule 1 — Correlation < 0.10 → DROP: {low_corr if low_corr else 'None'}")

# Rule 2: consistently bottom 3 across all methods
bottom3 = consensus[consensus['avg_rank'] > (len(CANDIDATE_FEATURES) - 3)].index.tolist()
print(f"Rule 2 — Consistently bottom 3  → DROP: {bottom3 if bottom3 else 'None'}")

# Combine drop list
to_drop = list(set(low_corr + bottom3))
print(f"\nTotal features to drop: {to_drop if to_drop else 'None'}")

# Selected features = candidates minus dropped
selected_features = [f for f in CANDIDATE_FEATURES if f not in to_drop]

print(f"\n── Selected Features ({len(selected_features)}) ──")
for i, f in enumerate(selected_features, 1):
    corr_val = corr_scores[f]
    print(f"  {i:2d}. {f:<30}  (corr={corr_val:.4f})")

── Elimination Analysis ──

Rule 1 — Correlation < 0.10 → DROP: ['participation_score', 'semester_number']
Rule 2 — Consistently bottom 3  → DROP: ['attendance_tier', 'semester_number']

Total features to drop: ['participation_score', 'attendance_tier', 'semester_number']

── Selected Features (11) ──
   1. attendance_percentage           (corr=0.3175)
   2. quiz_score_avg                  (corr=0.8163)
   3. assignment_score_avg            (corr=0.8177)
   4. midterm_score                   (corr=0.8187)
   5. historical_gpa                  (corr=0.6718)
   6. study_hours_per_week            (corr=0.5581)
   7. subject_difficulty_score        (corr=0.1414)
   8. gpa_tier                        (corr=0.6055)
   9. ca_avg                          (corr=0.8667)
  10. midterm_relative                (corr=0.7533)
  11. rule_risk_score                 (corr=0.8565)


## 9. Multicollinearity Check
Flag any pair of selected features with correlation > 0.90

In [13]:
corr_matrix = df[selected_features].corr()

print("── Multicollinearity Check (pairs with |r| > 0.90) ──")
high_corr_pairs = []
for i in range(len(selected_features)):
    for j in range(i + 1, len(selected_features)):
        f1 = selected_features[i]
        f2 = selected_features[j]
        r  = abs(corr_matrix.loc[f1, f2])
        if r > 0.90:
            high_corr_pairs.append((f1, f2, round(r, 4)))
            print(f"  ⚠️  {f1}  ↔  {f2}  :  r = {r:.4f}")

if not high_corr_pairs:
    print("  ✅ No multicollinear pairs found above 0.90")

# Heatmap of selected features only
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    ax=ax,
    vmin=-1, vmax=1
)
ax.set_title('Correlation Matrix — Selected Features', fontweight='bold')
plt.tight_layout()
plt.savefig('../eda/selected_features_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved → eda/selected_features_correlation.png")

── Multicollinearity Check (pairs with |r| > 0.90) ──
  ⚠️  quiz_score_avg  ↔  ca_avg  :  r = 0.9411
  ⚠️  assignment_score_avg  ↔  ca_avg  :  r = 0.9442
  ⚠️  midterm_score  ↔  midterm_relative  :  r = 0.9066

Plot saved → eda/selected_features_correlation.png


## 10. Handle Multicollinear Pairs (if any)

In [14]:
# For each flagged pair, drop the one with the lower avg_rank (weaker feature)
features_to_remove_mc = []

for f1, f2, r in high_corr_pairs:
    rank_f1 = consensus.loc[f1, 'avg_rank']
    rank_f2 = consensus.loc[f2, 'avg_rank']
    weaker  = f1 if rank_f1 > rank_f2 else f2
    stronger = f2 if weaker == f1 else f1
    features_to_remove_mc.append(weaker)
    print(f"  Keeping '{stronger}' (avg_rank={min(rank_f1,rank_f2):.2f}), "
          f"dropping '{weaker}' (avg_rank={max(rank_f1,rank_f2):.2f})")

if features_to_remove_mc:
    selected_features = [f for f in selected_features
                         if f not in features_to_remove_mc]
    print(f"\nFeatures removed due to multicollinearity: {features_to_remove_mc}")
else:
    print("  No features removed — no multicollinear pairs above 0.90")

# ── Manually remove gpa_tier ──────────────────────────────────────────────────
# gpa_tier is redundant with historical_gpa (same information, just binned).
# historical_gpa ranked better (avg_rank 8.00 vs 10.00) so we keep historical_gpa.
if 'gpa_tier' in selected_features:
    selected_features = [f for f in selected_features if f != 'gpa_tier']
    print(f"\n  Also dropping 'gpa_tier' — redundant with 'historical_gpa'")

print(f"\n── FINAL Selected Features ({len(selected_features)}) ──")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")

  Keeping 'ca_avg' (avg_rank=1.00), dropping 'quiz_score_avg' (avg_rank=6.00)
  Keeping 'ca_avg' (avg_rank=1.00), dropping 'assignment_score_avg' (avg_rank=5.00)
  Keeping 'midterm_score' (avg_rank=4.00), dropping 'midterm_relative' (avg_rank=6.33)

Features removed due to multicollinearity: ['quiz_score_avg', 'assignment_score_avg', 'midterm_relative']

  Also dropping 'gpa_tier' — redundant with 'historical_gpa'

── FINAL Selected Features (7) ──
   1. attendance_percentage
   2. midterm_score
   3. historical_gpa
   4. study_hours_per_week
   5. subject_difficulty_score
   6. ca_avg
   7. rule_risk_score


## 11. Save Feature List as PKL

In [15]:
pkl_path = '../models/feature_list.pkl'
joblib.dump(selected_features, pkl_path)

# Verify by reloading
loaded_features = joblib.load(pkl_path)

print("── feature_list.pkl saved and verified ──")
print(f"  Path    : {pkl_path}")
print(f"  Count   : {len(loaded_features)} features")
print(f"  Match   : {'✅ YES' if loaded_features == selected_features else '❌ MISMATCH'}")
print(f"  Content : {loaded_features}")

── feature_list.pkl saved and verified ──
  Path    : ../models/feature_list.pkl
  Count   : 7 features
  Match   : ✅ YES
  Content : ['attendance_percentage', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'subject_difficulty_score', 'ca_avg', 'rule_risk_score']


## 12. Save Selected Dataset

In [16]:
save_cols   = selected_features + ['final_score', 'pass_fail']
df_selected = df[save_cols]

out_path = '../data/features/features_selectionv2.csv'
df_selected.to_csv(out_path, index=False)

print("── features_selectionv2.csv saved ──")
print(f"  Path    : {out_path}")
print(f"  Shape   : {df_selected.shape}")
print(f"  Columns : {df_selected.columns.tolist()}")

── features_selectionv2.csv saved ──
  Path    : ../data/features/features_selectionv2.csv
  Shape   : (5000, 9)
  Columns : ['attendance_percentage', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'subject_difficulty_score', 'ca_avg', 'rule_risk_score', 'final_score', 'pass_fail']


## 13. Final Summary

In [17]:
print("╔══════════════════════════════════════════════════════════╗")
print("║           FEATURE SELECTION SUMMARY                     ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Candidate features          : {len(CANDIDATE_FEATURES):<5}                    ║")
print(f"║  Features dropped (low corr) : {len(low_corr):<5}                    ║")
print(f"║  Features dropped (bottom 3) : {len(bottom3):<5}                    ║")
print(f"║  Features dropped (multi-col): {len(features_to_remove_mc):<5}                    ║")
print(f"║  FINAL selected features     : {len(selected_features):<5}                    ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Final Feature List:                                     ║")
for i, f in enumerate(selected_features, 1):
    print(f"║    {i:2d}. {f:<50}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Saved:                                                  ║")
print("║    models/feature_list.pkl                               ║")
print("║    data/features/dataset_v1_selected.csv                 ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Next step → 06_model_training.ipynb                     ║")
print("╚══════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════════╗
║           FEATURE SELECTION SUMMARY                     ║
╠══════════════════════════════════════════════════════════╣
║  Candidate features          : 14                       ║
║  Features dropped (low corr) : 2                        ║
║  Features dropped (bottom 3) : 2                        ║
║  Features dropped (multi-col): 3                        ║
║  FINAL selected features     : 7                        ║
╠══════════════════════════════════════════════════════════╣
║  Final Feature List:                                     ║
║     1. attendance_percentage                             ║
║     2. midterm_score                                     ║
║     3. historical_gpa                                    ║
║     4. study_hours_per_week                              ║
║     5. subject_difficulty_score                          ║
║     6. ca_avg                                            ║
║     7. rule_risk_score      